In [ ]:
from torch.utils.data import Dataset, DataLoader, random_split
import pandas as pd
import ast
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import numpy as np
from sklearn.metrics import f1_score

## Dataset

In [60]:
train = pd.read_parquet("../data/train.parquet")
display(train.head())

test = pd.read_csv("../data/test.csv")
display(test.head())

,id,content,lang,manipulative,techniques,trigger_words
0,0bb0c7fa-101b-4583-a5f9-9d503339141c,Новий огляд мапи DeepState від російського вій...,uk,True,"[euphoria, loaded_language]","[[27, 63], [65, 88], [90, 183], [186, 308]]"
1,7159f802-6f99-4e9d-97bd-6f565a4a0fae,Недавно 95 квартал жёстко поглумился над русск...,ru,True,"[loaded_language, cherry_picking]","[[0, 40], [123, 137], [180, 251], [253, 274]]"
2,e6a427f1-211f-405f-bd8b-70798458d656,🤩\nТим часом йде евакуація Бєлгородського авто...,uk,True,"[loaded_language, euphoria]","[[55, 100]]"
3,1647a352-4cd3-40f6-bfa1-d87d42e34eea,В Україні найближчим часом мають намір посилит...,uk,False,None,None
4,9c01de00-841f-4b50-9407-104e9ffb03bf,"Расчёты 122-мм САУ 2С1 ""Гвоздика"" 132-й бригад...",ru,True,[loaded_language],"[[114, 144]]"


,id,content,trigger_words,techniques
0,521cd2e8-dd9f-42c4-98ba-c0c8890ff1ba,"Они просрали нашу технику, положили кучу людей...","[(0, 12), (27, 46), (48, 71), (131, 162), (164...","['fud', 'loaded_language']"
1,9b2a61e4-d14e-4ff7-b304-e73d720319bf,❗️\nКитай предлагает отдать оккупированные тер...,"[(374, 425)]",['loaded_language']
2,f0f1c236-80a8-4d25-b30c-a420a39be632,Сегодня будет ровно 6 месяцев с этого обещания...,"[(0, 127)]",['loaded_language']
3,31ea05ba-2c2b-4b84-aba7-f3cf6841b204,⚡️\nІзраїль вперше у світі збив балістичну рак...,NaN,[]
4,a79e13ec-6d9a-40b5-b54c-7f4f743a7525,Склав невелику навчально-методичну таблицю на ...,"[(87, 103), (127, 136), (170, 189), (204, 255)...",['loaded_language']


In [61]:
train['techniques'] = train['techniques'].apply(lambda x: [] if x is None else x)
train['trigger_words'] = train['trigger_words'].apply(lambda x: [] if x is None else x)

In [62]:
Counter([tech for sublist in train['techniques'] for tech in sublist])

Counter({'loaded_language': 1973,
         'cherry_picking': 512,
         'glittering_generalities': 483,
         'cliche': 463,
         'euphoria': 462,
         'fud': 385,
         'appeal_to_fear': 300,
         'whataboutism': 158,
         'bandwagon': 157,
         'straw_man': 138})

In [63]:
test['trigger_words'] = test['trigger_words'].apply(lambda x: x if isinstance(x, str) else "[]")
test['trigger_words'] = test['trigger_words'].apply(ast.literal_eval)
test['techniques'] = test['techniques'].apply(ast.literal_eval)

In [64]:
Counter([tech for sublist in test['techniques'] for tech in sublist])

Counter({'loaded_language': 2959,
         'cherry_picking': 768,
         'glittering_generalities': 723,
         'euphoria': 695,
         'cliche': 695,
         'fud': 576,
         'appeal_to_fear': 449,
         'bandwagon': 236,
         'whataboutism': 235,
         'straw_man': 207})

## Task 1 (f1 scores for multilabel)

In [41]:
best_multilabel = pd.read_csv("../data/gemma_LM_CLS_advanced.csv")
second_best_multilabel = pd.read_csv("../data/gemma_LM_CLS.csv")
baseline_multilabel = pd.read_csv("../data/zero_shot_catboost_multilabel.csv")

In [42]:
best_multilabel['best_predicted_techniques'] = best_multilabel.apply(lambda row: [col for col in best_multilabel.columns if row[col] == 1], axis=1)
second_best_multilabel['second_best_predicted_techniques'] = second_best_multilabel.apply(lambda row: [col for col in second_best_multilabel.columns if row[col] == 1], axis=1)
baseline_multilabel['baseline_predicted_techniques'] = baseline_multilabel.apply(lambda row: [col for col in baseline_multilabel.columns if row[col] == 1], axis=1)

In [43]:
test_preds = test.merge(best_multilabel[['id', 'best_predicted_techniques']], on="id", how="left")

In [44]:
test_preds = test_preds.merge(second_best_multilabel[['id', 'second_best_predicted_techniques']], on="id", how="left")

In [45]:
test_preds = test_preds.merge(baseline_multilabel[['id', 'baseline_predicted_techniques']], on="id", how="left")

In [46]:
test_preds.head()

,id,content,trigger_words,techniques,best_predicted_techniques,second_best_predicted_techniques,baseline_predicted_techniques
0,521cd2e8-dd9f-42c4-98ba-c0c8890ff1ba,"Они просрали нашу технику, положили кучу людей...","[(0, 12), (27, 46), (48, 71), (131, 162), (164...","[fud, loaded_language]","[loaded_language, cliche, fud]","[loaded_language, cliche, fud]","[loaded_language, cherry_picking, fud]"
1,9b2a61e4-d14e-4ff7-b304-e73d720319bf,❗️\nКитай предлагает отдать оккупированные тер...,"[(374, 425)]",[loaded_language],[loaded_language],[loaded_language],"[loaded_language, cherry_picking, appeal_to_fear]"
2,f0f1c236-80a8-4d25-b30c-a420a39be632,Сегодня будет ровно 6 месяцев с этого обещания...,"[(0, 127)]",[loaded_language],[loaded_language],[loaded_language],[loaded_language]
3,31ea05ba-2c2b-4b84-aba7-f3cf6841b204,⚡️\nІзраїль вперше у світі збив балістичну рак...,[],[],[],[],[]
4,a79e13ec-6d9a-40b5-b54c-7f4f743a7525,Склав невелику навчально-методичну таблицю на ...,"[(87, 103), (127, 136), (170, 189), (204, 255)...",[loaded_language],"[loaded_language, cliche, bandwagon]","[loaded_language, cliche, whataboutism]","[loaded_language, cliche, whataboutism]"


In [48]:
from sklearn.preprocessing import MultiLabelBinarizer

# Labels you are working with
all_labels = sorted(set(label for labels in test_preds['techniques'] for label in labels +
                        test_preds['best_predicted_techniques'].explode().dropna().tolist() +
                        test_preds['second_best_predicted_techniques'].explode().dropna().tolist()))

mlb = MultiLabelBinarizer(classes=all_labels)

# Transform to binary matrices
y_true = mlb.fit_transform(test_preds['techniques'])
y_pred_best = mlb.transform(test_preds['best_predicted_techniques'])
y_pred_2best = mlb.transform(test_preds['second_best_predicted_techniques'])
y_pred_baseline = mlb.transform(test_preds['baseline_predicted_techniques'])

In [49]:
f1_baseline = f1_score(y_true, y_pred_baseline, average="macro")
f1_baseline

0.40801727349240496

In [50]:
f1_2best = f1_score(y_true, y_pred_2best, average="macro")
f1_2best

0.45007843808725073

In [51]:
f1_best = f1_score(y_true, y_pred_best, average="macro")
f1_best

0.4544742063751106

## Task 2

In [68]:
import pandas as pd
import pandas.api.types
from sklearn.metrics import f1_score
import ast


class ParticipantVisibleError(Exception):
    """Custom exception for participant-visible errors."""
    pass


def score(solution: pd.DataFrame, submission: pd.DataFrame, row_id_column_name: str) -> float:
    """
    Compute span-level F1 score based on overlap.

    Parameters:
    - solution (pd.DataFrame): Ground truth DataFrame with row ID and token labels.
    - submission (pd.DataFrame): Submission DataFrame with row ID and token labels.
    - row_id_column_name (str): Column name for the row identifier.

    Returns:
    - float: The token-level weighted F1 score.

    Example:
    >>> solution = pd.DataFrame({
    ...     "id": [1, 2, 3],
    ...     "trigger_words": [[(612, 622), (725, 831)], [(300, 312)], []]
    ... })
    >>> submission = pd.DataFrame({
    ...     "id": [1, 2, 3],
    ...     "trigger_words": [[(612, 622), (700, 720)], [(300, 312)], [(100, 200)]]
    ... })
    >>> score(solution, submission, "id")
    0.16296296296296295
    """
    if not all(col in solution.columns for col in ["id", "trigger_words"]):
        raise ValueError("Solution DataFrame must contain 'id' and 'trigger_words' columns.")
    if not all(col in submission.columns for col in ["id", "trigger_words"]):
        raise ValueError("Submission DataFrame must contain 'id' and 'trigger_words' columns.")
    
    def safe_parse_spans(trigger_words):
        if isinstance(trigger_words, str):
            try:
                return ast.literal_eval(trigger_words)
            except (ValueError, SyntaxError):
                return []
        if isinstance(trigger_words, (list, tuple)):
            return trigger_words
        return []

    def extract_tokens_from_spans(spans):
        tokens = set()
        for start, end in spans:
            tokens.update(range(start, end))
        return tokens
    
    solution = solution.copy()
    submission = submission.copy()

    solution["trigger_words"] = solution["trigger_words"].apply(safe_parse_spans)
    submission["trigger_words"] = submission["trigger_words"].apply(safe_parse_spans)

    merged = pd.merge(
        solution,
        submission,
        on="id",
        suffixes=("_solution", "_submission")
    )

    total_true_tokens = 0
    total_pred_tokens = 0
    overlapping_tokens = 0

    for _, row in merged.iterrows():
        true_spans = row["trigger_words_solution"]
        pred_spans = row["trigger_words_submission"]

        true_tokens = extract_tokens_from_spans(true_spans)
        pred_tokens = extract_tokens_from_spans(pred_spans)

        total_true_tokens += len(true_tokens)
        total_pred_tokens += len(pred_tokens)
        overlapping_tokens += len(true_tokens & pred_tokens)

    precision = overlapping_tokens / total_pred_tokens if total_pred_tokens > 0 else 0
    recall = overlapping_tokens / total_true_tokens if total_true_tokens > 0 else 0
    f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

    return f1

In [65]:
best_token = pd.read_csv("../data/pred_token_multitarget_baseline (1).csv")
best_token

,id,trigger_words
0,521cd2e8-dd9f-42c4-98ba-c0c8890ff1ba,"[(0, 253)]"
1,9b2a61e4-d14e-4ff7-b304-e73d720319bf,"[(20, 52), (58, 74), (341, 342), (374, 387), (..."
2,f0f1c236-80a8-4d25-b30c-a420a39be632,"[(0, 7), (14, 127), (142, 143)]"
3,31ea05ba-2c2b-4b84-aba7-f3cf6841b204,[]
4,a79e13ec-6d9a-40b5-b54c-7f4f743a7525,"[(6, 8), (13, 14), (60, 63), (67, 72), (87, 10..."
...,...,...
5730,e8e22b6d-0068-4afb-b606-4a1baa8a8d4c,"[(0, 4), (39, 60), (113, 145), (147, 197), (19..."
5731,8b1d69b4-69ce-4e40-b4ba-dd2f370a8b6f,"[(3, 4), (12, 15), (17, 104), (175, 238), (240..."
5732,c2246217-3358-4f61-bda8-e2ec21aed5b2,"[(384, 386), (401, 423), (444, 446), (471, 475)]"
5733,45aa63c4-2248-4a0e-8f66-f3d23b6828ed,"[(0, 68), (71, 72), (83, 84), (227, 248), (250..."


In [66]:
best_token['trigger_words'] = best_token['trigger_words'].apply(ast.literal_eval)

In [69]:
f1 = score(best_token, test, row_id_column_name="id")
print("Span-level F1 score:", f1)

Span-level F1 score: 0.5988803556149175


In [72]:
baseline_token = pd.read_csv("../data/pred_token_baseline.csv")
baseline_token

,id,trigger_words
0,521cd2e8-dd9f-42c4-98ba-c0c8890ff1ba,"[(0, 253)]"
1,9b2a61e4-d14e-4ff7-b304-e73d720319bf,"[(20, 26), (32, 52), (53, 74), (172, 188), (32..."
2,f0f1c236-80a8-4d25-b30c-a420a39be632,"[(0, 0), (20, 127), (142, 143)]"
3,31ea05ba-2c2b-4b84-aba7-f3cf6841b204,[]
4,a79e13ec-6d9a-40b5-b54c-7f4f743a7525,"[(0, 0), (24, 25), (87, 103), (127, 309)]"
...,...,...
5730,e8e22b6d-0068-4afb-b606-4a1baa8a8d4c,"[(0, 0), (32, 33), (113, 261), (367, 368), (42..."
5731,8b1d69b4-69ce-4e40-b4ba-dd2f370a8b6f,"[(0, 0), (3, 4), (15, 105), (174, 497)]"
5732,c2246217-3358-4f61-bda8-e2ec21aed5b2,"[(382, 480)]"
5733,45aa63c4-2248-4a0e-8f66-f3d23b6828ed,"[(0, 4), (67, 70), (259, 274)]"


In [73]:
baseline_token['trigger_words'] = baseline_token['trigger_words'].apply(ast.literal_eval)

In [74]:
f1 = score(baseline_token, test, row_id_column_name="id")
print("Span-level F1 score:", f1)

Span-level F1 score: 0.5858810450250138


In [70]:
true_span_counts = []
pred_span_counts = []

for true_spans, pred_spans in zip(test['trigger_words'] , best_token['trigger_words'] ):
    true_span_counts.append(len(true_spans))
    pred_span_counts.append(len(pred_spans))

average_true_spans = sum(true_span_counts) / len(true_span_counts)
average_pred_spans = sum(pred_span_counts) / len(pred_span_counts)

print(f"Average number of true spans per sample: {average_true_spans:.2f}")
print(f"Average number of predicted spans per sample: {average_pred_spans:.2f}")


Average number of true spans per sample: 2.29
Average number of predicted spans per sample: 4.86


In [ ]:
true_span_counts = []
pred_span_counts = []

for true_spans, pred_spans in zip(test['trigger_words'],baseline_token['trigger_words'] ):
    true_span_counts.append(len(true_spans))
    pred_span_counts.append(len(pred_spans))

average_true_spans = sum(true_span_counts) / len(true_span_counts)
average_pred_spans = sum(pred_span_counts) / len(pred_span_counts)

print(f"Average number of true spans per sample: {average_true_spans:.2f}")
print(f"Average number of predicted spans per sample: {average_pred_spans:.2f}")

Average number of true spans per sample: 2.29
Average number of predicted spans per sample: 3.79


## Error analysis

In [5]:
best_multilabel = pd.read_csv("../data/gemma_LM_CLS_advanced.csv")

In [3]:
second_best_multilabel = pd.read_csv("../data/gemma_LM_CLS.csv")

In [10]:
best_multilabel['predicted_techniques'] = best_multilabel.apply(lambda row: [col for col in best_multilabel.columns if row[col] == 1], axis=1)
best_multilabel.head()

,euphoria,loaded_language,cherry_picking,glittering_generalities,cliche,appeal_to_fear,bandwagon,fud,whataboutism,straw_man,id,best_predicted_techniques,predicted_techniques
0,0,1,0,0,1,0,0,1,0,0,521cd2e8-dd9f-42c4-98ba-c0c8890ff1ba,"[loaded_language, cliche, fud]","[loaded_language, cliche, fud]"
1,0,1,0,0,0,0,0,0,0,0,9b2a61e4-d14e-4ff7-b304-e73d720319bf,[loaded_language],[loaded_language]
2,0,1,0,0,0,0,0,0,0,0,f0f1c236-80a8-4d25-b30c-a420a39be632,[loaded_language],[loaded_language]
3,0,0,0,0,0,0,0,0,0,0,31ea05ba-2c2b-4b84-aba7-f3cf6841b204,[],[]
4,0,1,0,0,1,0,1,0,0,0,a79e13ec-6d9a-40b5-b54c-7f4f743a7525,"[loaded_language, cliche, bandwagon]","[loaded_language, cliche, bandwagon]"


In [5]:
test_preds = test.merge(second_best_multilabel[['id', 'predicted_techniques']], on="id", how="left")

In [6]:
test_preds.head()

,id,content,trigger_words,techniques,predicted_techniques
0,521cd2e8-dd9f-42c4-98ba-c0c8890ff1ba,"Они просрали нашу технику, положили кучу людей...","[(0, 12), (27, 46), (48, 71), (131, 162), (164...","['fud', 'loaded_language']","[loaded_language, cliche, fud]"
1,9b2a61e4-d14e-4ff7-b304-e73d720319bf,❗️\nКитай предлагает отдать оккупированные тер...,"[(374, 425)]",['loaded_language'],[loaded_language]
2,f0f1c236-80a8-4d25-b30c-a420a39be632,Сегодня будет ровно 6 месяцев с этого обещания...,"[(0, 127)]",['loaded_language'],[loaded_language]
3,31ea05ba-2c2b-4b84-aba7-f3cf6841b204,⚡️\nІзраїль вперше у світі збив балістичну рак...,NaN,[],[]
4,a79e13ec-6d9a-40b5-b54c-7f4f743a7525,Склав невелику навчально-методичну таблицю на ...,"[(87, 103), (127, 136), (170, 189), (204, 255)...",['loaded_language'],"[loaded_language, cliche, whataboutism]"


In [53]:
from sklearn.metrics import classification_report
from sklearn.preprocessing import MultiLabelBinarizer


mlb = MultiLabelBinarizer()
y_true = mlb.fit_transform(test_preds['techniques'])
y_pred = mlb.transform(test_preds['second_best_predicted_techniques'])

print(classification_report(y_true, y_pred, target_names=mlb.classes_))

                         precision    recall  f1-score   support

         appeal_to_fear       0.40      0.59      0.48       449
              bandwagon       0.23      0.14      0.18       236
         cherry_picking       0.45      0.48      0.46       768
                 cliche       0.23      0.42      0.30       695
               euphoria       0.56      0.52      0.54       695
                    fud       0.52      0.59      0.55       576
glittering_generalities       0.67      0.60      0.63       723
        loaded_language       0.73      0.85      0.78      2959
              straw_man       0.23      0.39      0.29       207
           whataboutism       0.21      0.50      0.29       235

              micro avg       0.52      0.64      0.57      7543
              macro avg       0.42      0.51      0.45      7543
           weighted avg       0.55      0.64      0.59      7543
            samples avg       0.37      0.44      0.38      7543



/Users/k_akhynko/Desktop/work/thesis/manipulative-narrative-detection/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/k_akhynko/Desktop/work/thesis/manipulative-narrative-detection/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/k_akhynko/Desktop/work/thesis/manipulative-narrative-detection/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in samples with no true nor predi